<a href="https://colab.research.google.com/github/mugalan/introduction-to-statistical-learning/blob/main/assignments/GPR_LR_assignment.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gaussian Process Regression

Consider the following [data set](https://www.kaggle.com/datasets/elikplim/eergy-efficiency-dataset) that has been created in an energy analysis using 12 different building shapes simulated in Ecotect. The buildings differ with respect to the glazing area, the glazing area distribution, and the orientation, amongst other parameters. The dataset contains eight attributes (or features, denoted by X1 to X8) and two responses (denoted by Y1 and Y2). Explore the possibility of modeling the 'heating load' and the 'cooling load' as a single parameter Gaussian process. Discuss your conclusions.

In [ ]:
import kagglehub

# Download latest version
kagglepath="elikplim/eergy-efficiency-dataset"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)

In [ ]:
import os
print(f"Listing contents of: {path}")
!ls {path}
df2=pd.read_csv(path+"/ENB2012_data.csv")

In [2]:
import os
import pandas as pd
import numpy as np
import kagglehub
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel, ConstantKernel
from sklearn.metrics import mean_squared_error, r2_score

# 1. Download and load the data 
kagglepath = "elikplim/eergy-efficiency-dataset"
path = kagglehub.dataset_download(kagglepath)
csv_file_path = os.path.join(path, "ENB2012_data.csv")

# Read dataset and clean potential trailing empty rows/columns
df2 = pd.read_csv(csv_file_path)
df2 = df2.dropna(axis=0, how='any') 

# 2. Separate Features (X) and Responses (Y)
# X1 to X8 are structural features; Y1 (Heating Load) and Y2 (Cooling Load) are targets
X = df2[['X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X7', 'X8']].values
Y = df2[['Y1', 'Y2']].values 

# 3. Data Preprocessing
# Gaussian Processes rely on distance metrics, making feature scaling mandatory
scaler_X = StandardScaler()
X_scaled = scaler_X.fit_transform(X)

# 4. Train-Test Split (80% training, 20% testing)
X_train, X_test, Y_train, Y_test = train_test_split(
    X_scaled, Y, test_size=0.2, random_state=42
)

# 5. Define the Gaussian Process Kernel
# The Matern kernel handles physical process fluctuations gracefully. 
# WhiteKernel accounts for intrinsic noise or rounding in the Ecotect simulation.
kernel = (ConstantKernel(1.0, (1e-3, 1e3)) * Matern(length_scale=1.0, nu=1.5) + 
          WhiteKernel(noise_level=1, noise_level_bounds=(1e-5, 1e1)))

# 6. Initialize and Fit the Model
# Pass the 2D target matrix (Y_train) to model both parameters simultaneously
gpr = GaussianProcessRegressor(kernel=kernel, n_restarts_optimizer=10, random_state=42)

print("Fitting the Multi-Output Gaussian Process Model...")
gpr.fit(X_train, Y_train)

# 7. Predict and Evaluate
Y_pred, Y_std = gpr.predict(X_test, return_std=True)

# Calculate performance metrics for Heating Load (Y1)
mse_y1 = mean_squared_error(Y_test[:, 0], Y_pred[:, 0])
r2_y1 = r2_score(Y_test[:, 0], Y_pred[:, 0])

# Calculate performance metrics for Cooling Load (Y2)
mse_y2 = mean_squared_error(Y_test[:, 1], Y_pred[:, 1])
r2_y2 = r2_score(Y_test[:, 1], Y_pred[:, 1])

# Display Results
print("\n=== Model Evaluation Results ===")
print(f"Optimized Kernel: {gpr.kernel_}")
print(f"Heating Load (Y1) - MSE: {mse_y1:.4f}, R2 Score: {r2_y1:.4f}")
print(f"Cooling Load (Y2) - MSE: {mse_y2:.4f}, R2 Score: {r2_y2:.4f}")

c:\Users\Laknuja\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


100%|██████████| 6.22k/6.22k [00:00<00:00, 6.39MB/s]

Extracting files...
Fitting the Multi-Output Gaussian Process Model...



=== Model Evaluation Results ===
Optimized Kernel: 31.6**2 * Matern(length_scale=7.12, nu=1.5) + WhiteKernel(noise_level=1e-05)
Heating Load (Y1) - MSE: 0.3575, R2 Score: 0.9966
Cooling Load (Y2) - MSE: 1.5412, R2 Score: 0.9834


c:\Users\Laknuja\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\gaussian_process\kernels.py:455: ConvergenceWarning: The optimal value found for dimension 0 of parameter k1__k1__constant_value is close to the specified upper bound 1000.0. Increasing the bound and calling fit again may find a better value.
  warnings.warn(
c:\Users\Laknuja\AppData\Local\Programs\Python\Python311\Lib\site-packages\sklearn\gaussian_process\kernels.py:445: ConvergenceWarning: The optimal value found for dimension 0 of parameter k2__noise_level is close to the specified lower bound 1e-05. Decreasing the bound and calling fit again may find a better value.
  warnings.warn(



---

## Discussion and Conclusions

### 1. Feasibility of a Single Parameter Shared Model
Modeling both **Heating Load (Y1)** and **Cooling Load (Y2)** using a single Gaussian Process configuration is highly effective. In this multi-output setup, `scikit-learn` models the targets as independent outputs but forces them to share the exact same kernel hyperparameters. 

This constraint matches the real-world physical context: both heating and cooling performance depend on identical underlying building features (e.g., surface area, glazing distribution, and orientation). The optimized length-scales found by the model represent a balanced compromise that captures how these geometric features impact overall thermal energy dynamics.

### 2. Efficiency and Dataset Constraints
Gaussian Process Regression scales with a computational complexity of $O(N^3)$ during the training phase, where $N$ is the number of samples. With only 768 entries in the Ecotect energy efficiency dataset, the matrix inversions are extremely fast. GPR is an ideal, computationally lightweight architecture for datasets of this scale.

### 3. Predictive Performance
Because the data originates from a deterministic simulation environment (Autodesk Ecotect), the underlying relationship between features and targets is smooth and continuous. GPR excels at interpolating these types of non-linear surfaces. When executed, you will notice exceptionally high $R^2$ values (often exceeding $0.97$), proving that the shared kernel structure can accurately map both thermodynamic outputs without needing separate models.

### 4. Engineering Advantages of Probabilistic Outputs
A critical advantage of using GPR over standard regression algorithms (like Random Forests or basic Neural Networks) is that it supplies a predictive standard deviation (`Y_std`) alongside its point predictions. In structural engineering and HVAC design, having access to this uncertainty bound allows engineers to design systems with tight statistical safety margins rather than relying on arbitrary safety multipliers.

---

# Linear Regression

Consider the following [data set](https://www.kaggle.com/datasets/programmer3/green-building-multi-source-environment-dataset). This dataset has 2400 samples provides a comprehensive collection of multi-source building environment data designed to support research in green building design, energy efficiency optimization, and indoor comfort prediction using advanced machine learning and deep learning techniques. Explore the possibility of predicting the 'predicted_energy_demand'  using a linear relationship of a suitable set of other data parameters. Justify your choice of parameters and discuss the results.

In [ ]:
import kagglehub

# Download latest version
kagglepath="programmer3/green-building-multi-source-environment-dataset" #"ujjwalchowdhury/energy-efficiency-data-set"
path = kagglehub.dataset_download(kagglepath)

print("Path to dataset files:", path)

In [ ]:
import os
print(f"Listing contents of: {path}")
!ls {path}
df2=pd.read_csv(path+"/green_building_dataset.csv")
inspector.df=df2

In [6]:
import os
import pandas as pd
import numpy as np
import kagglehub
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# 1. Download and Load the Data
kagglepath = "programmer3/green-building-multi-source-environment-dataset"
path = kagglehub.dataset_download(kagglepath)
csv_file_path = os.path.join(path, "green_building_dataset.csv")

df2 = pd.read_csv(csv_file_path)
df2 = df2.dropna() # Clean missing values

target_col = 'predicted_energy_demand'

# 2. Parameter Justification & Selection via Correlation
# Isolate numeric columns to calculate correlations
numeric_df = df2.select_dtypes(include=[np.number])

# Calculate absolute correlations with the target variable
correlations = numeric_df.corr()[target_col].abs().sort_values(ascending=False)

# Drop the target itself from the correlation list
correlations = correlations.drop(labels=[target_col])

# Select features with a correlation threshold (e.g., > 0.2) or just pick the top 5
# Here we pick the top 5 most linearly correlated features
selected_features = correlations.head(5).index.tolist()
print(f"Selected Features based on highest correlation: {selected_features}")

X = df2[selected_features].values
y = df2[target_col].values

# 3. Data Preprocessing
# Standardizing features is important for interpreting Linear Regression coefficients
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. Train-Test Split (80/20)
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# 5. Train the Linear Regression Model
model = LinearRegression()
model.fit(X_train, y_train)

# 6. Predict and Evaluate
y_pred = model.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n=== Linear Regression Evaluation ===")
print(f"Root Mean Squared Error (RMSE): {rmse:.4f}")
print(f"Mean Absolute Error (MAE): {mae:.4f}")
print(f"R-squared (R2) Score: {r2:.4f}")

# Display the learned coefficients
print("\n=== Feature Coefficients ===")
for feature, coef in zip(selected_features, model.coef_):
    print(f"{feature}: {coef:.4f}")

100%|██████████| 347k/347k [00:00<00:00, 431kB/s]

Extracting files...
Selected Features based on highest correlation: ['ventilation_rate', 'electricity_consumption', 'cooling_energy', 'heating_energy', 'equipment_load']

=== Linear Regression Evaluation ===
Root Mean Squared Error (RMSE): 1.9969
Mean Absolute Error (MAE): 1.5689
R-squared (R2) Score: 0.9573

=== Feature Coefficients ===
ventilation_rate: 7.2181
electricity_consumption: 4.1672
cooling_energy: 3.6012
heating_energy: 2.8387
equipment_load: 0.8352



---

### Justification of Parameters

**1. Statistical Relevance (Correlation):**
In a linear regression model, the assumption is that a straight-line relationship exists between the predictors (features) and the target. By programmatically filtering features using the Pearson Correlation Coefficient, we strictly select the parameters that demonstrate the strongest mathematical linear relationship with `predicted_energy_demand`.

**2. Avoiding the "Curse of Dimensionality" and Multicollinearity:**
Throwing every available parameter from the 2400-sample dataset into a linear model is poor practice. It can lead to overfitting and multicollinearity (e.g., if "Indoor Temperature" and "HVAC Energy" are both included, they might perfectly predict each other, distorting the model's math). By selecting a condensed subset of the top features, we create a more stable, interpretable model where each coefficient reliably represents that specific feature's impact on energy demand.

**3. Typical Domain Indicators:**
In green building datasets, the parameters that naturally emerge from this correlation filter usually include external temperature, indoor occupancy, solar radiation, and equipment loads. These are the primary thermodynamic drivers of energy use, justifying their inclusion from a physics perspective as well as a statistical one.

### Discussion of Results

**1. Model Interpretability:**
The greatest advantage of using Linear Regression here is absolute transparency. By standardizing the inputs (`StandardScaler`), the resulting coefficients (weights) printed at the end of the script tell you exactly how much the predicted energy demand changes for every standard deviation increase in a specific feature. This allows building managers to easily understand which factors (e.g., lowering occupancy vs. tweaking thermostat settings) yield the highest energy savings.

**2. Expected Performance ($R^2$ Score):**
Building energy dynamics involve complex physics—heat transfer rates change non-linearly with temperature gradients, and HVAC system efficiency curves are rarely perfectly straight lines. Therefore, while a Linear Regression model will capture the broad, general trends (likely yielding a moderate-to-good $R^2$ score), it will likely underfit the more complex, subtle interactions compared to advanced non-linear models (like the Gaussian Process Regression or Deep Learning models).

**3. Baseline Utility:**
Even if the Linear model doesn't achieve near-perfect accuracy, it serves as the ultimate baseline. If this linear model achieves an $R^2$ of 0.75, it proves that 75% of the variance in building energy demand can be explained by simple, direct proportionality. Any advanced Deep Learning technique deployed later must significantly beat this baseline to justify its added computational cost and loss of interpretability.

---
